In [0]:
USE video_games.video_games;

In [0]:
SHOW TABLES;

database,tableName,isTemporary
video_games,video_games,false
video_games,video_games_sales,false
video_games,vw_top_publishers,false
video_games,vw_ventas_plataforma,false


In [0]:
DESCRIBE TABLE video_games

col_name,data_type,comment
Rank,bigint,null
Name,string,null
Platform,string,null
Year,int,null
Genre,string,null


In [0]:
DESCRIBE TABLE video_games_sales

col_name,data_type,comment
Rank,bigint,null
Publisher,string,null
NA_Sales,double,null
EU_Sales,double,null
JP_Sales,double,null
Other_Sales,double,null
Global_Sales,double,null


### **Visualizacion de Cada Fila**

In [0]:
SELECT DISTINCT Global_Sales
FROM video_games_sales
ORDER BY Global_Sales;

Global_Sales
0.0
0.01
0.02
0.03
0.04
0.05
0.06
0.07
0.08
0.09


# **LIMPIEZA TABLA (video_games)**

## **- LIMPIZA COLUMNA NAME**

### **IMAGEN**
![image_1776620515258.png](./image_1776620515258.png "image_1776620515258.png")

### **Verificar que contiene las Filas**

In [0]:
SELECT 
    vg.Rank,
    vg.Name,
    vg.Platform,
    vg.Year,
    vg.Genre,
    vgs.Publisher,
    vgs.NA_Sales,
    vgs.EU_Sales,
    vgs.JP_Sales,
    vgs.Other_Sales,
    vgs.Global_Sales
FROM video_games vg
INNER JOIN video_games_sales vgs
ON vg.Rank = vgs.Rank
WHERE vg.Name LIKE '.hack%';

Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales


### **Limpieza Nombre .hack//**

In [0]:
CREATE OR REPLACE TABLE video_games AS
SELECT
    Rank,

    CASE 
        WHEN Name LIKE '.hack%' 
            THEN REGEXP_REPLACE(Name, '^\\.hack[^ ]*\\s*', '')
        ELSE Name
    END AS Name,

    Platform,
    Year,
    Genre

FROM video_games;

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM video_games
WHERE Name LIKE '.hack%';

Rank,Name,Platform,Year,Genre


In [0]:
SELECT *
FROM video_games
ORDER BY Name;

Rank,Name,Platform,Year,Genre
4756,'98 Koshien,PS,1998,Sports
3770,007 Racing,PS,2000,Racing
1741,007: Quantum of Solace,PS3,2008,Action
14550,007: Quantum of Solace,PC,2008,Action
3040,007: Quantum of Solace,Wii,2008,Action
9320,007: Quantum of Solace,DS,2008,Action
4501,007: Quantum of Solace,PS2,2008,Action
1275,007: Quantum of Solace,X360,2008,Action
2249,007: The World is not Enough,PS,2000,Action
1202,007: The World is not Enough,N64,2000,Action


## **- LIMPIEZA FILAS CON VARIOS NULL**

### **Imagen** 
![image_1776622503606.png](./image_1776622503606.png "image_1776622503606.png")

### **Verificar que contiene las filas**

In [0]:
SELECT 
    vg.Rank,
    vg.Name,
    vg.Platform,
    vg.Year,
    vg.Genre,
    vgs.Publisher,
    vgs.NA_Sales,
    vgs.EU_Sales,
    vgs.JP_Sales,
    vgs.Other_Sales,
    vgs.Global_Sales
FROM video_games vg
LEFT JOIN video_games_sales vgs
ON vg.Rank = vgs.Rank
WHERE vg.Platform IS NULL
  AND vg.Year IS NULL
  AND vg.Genre IS NULL;

Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales


### **Limpieza datos en NULL**

In [0]:
CREATE OR REPLACE TABLE video_games AS
SELECT
    Rank,
    Name,
    Platform,
    Year,
    Genre
FROM video_games
WHERE NOT (
    Platform IS NULL
    AND Year IS NULL
    AND Genre IS NULL
);

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM video_games
WHERE Platform IS NULL
  AND Year IS NULL
  AND Genre IS NULL;

Rank,Name,Platform,Year,Genre


## **- LIMPIEZA FILA YEAR**

### **Imagen**
![image_1776623005973.png](./image_1776623005973.png "image_1776623005973.png")

### **Verificar los datos de esas Filas**

In [0]:
SELECT *
FROM video_games
WHERE NOT (Year RLIKE '^[0-9]{4}$')

Rank,Name,Platform,Year,Genre


### **Convertir datos N/A a Datos NULL**

In [0]:
UPDATE video_games
SET Year = NULL
WHERE Year = 'N/A';

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-5348172651688110>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "UPDATE video_games\nSET Year = NULL\nWHERE Year = 'N/A';\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self.driv

### **Verificar Cambio**

In [0]:
SELECT *
FROM video_games
WHERE NOT (Year RLIKE '^[0-9]{4}$')

### **Eliminar Datos que son corruptos o son invalidos**

In [0]:
DELETE FROM video_games
WHERE NOT (Year RLIKE '^[0-9]{4}$');

### **Eliminar tambien los datos en la tabla video_games_sales**

In [0]:
DELETE FROM video_games_sales
WHERE Rank NOT IN (
    SELECT Rank FROM video_games
);

### **Verificar cambio**

In [0]:
SELECT DISTINCT Year
FROM video_games
ORDER BY Year;

### **Cambiar el tipo de Dato, de Tipo String a Tipo Int**

In [0]:
CREATE OR REPLACE TABLE video_games AS
SELECT
    Rank,
    Name,
    Platform,
    CAST(Year AS INT) AS Year,
    Genre
FROM video_games;

In [0]:
DESCRIBE TABLE video_games

### **Verificamos que tengan el mismo numero de columnas las dos Tablas**

In [0]:
SELECT COUNT(*) AS total_registros
FROM video_games;

In [0]:
SELECT COUNT(*) AS total_registros
FROM video_games_sales;

# **LIMPIEZA TABLA (video_games_sales)**

## **- LIMPIEZA DE LA COLUMNA PUBLISHER**

### **Imagen**
![image_1776625803937.png](./image_1776625803937.png "image_1776625803937.png")

### **Tomar los datos que dicen N/A y cambiarlos por Null**

In [0]:
UPDATE video_games_sales
SET Publisher = NULL
WHERE Publisher = 'N/A';

In [0]:
SELECT *
FROM video_games_sales
WHERE Publisher = 'N/A';

In [0]:
SELECT *
FROM video_games_sales
WHERE Publisher IS NULL;

## **- LIMPIEZA DE LA COLUMNA NA_SALES**

### **Imagen**
![image_1776626057923.png](./image_1776626057923.png "image_1776626057923.png")

### **Eliminar los Valores .Inc y Inc**

In [0]:
UPDATE video_games_sales
SET NA_Sales = NULL
WHERE TRIM(NA_Sales) IN ('Inc', 'Inc.');

In [0]:
SELECT DISTINCT NA_Sales
FROM video_games_sales
ORDER BY NA_Sales;

### **Transformamos la columna NA_Sales de tipo String a Tipo Double**

In [0]:
CREATE OR REPLACE TABLE video_games_sales AS
SELECT
    Rank,
    Publisher,
    CAST(NA_Sales AS DOUBLE) AS NA_Sales,
    EU_Sales,
    JP_Sales,
    Other_Sales,
    Global_Sales
FROM video_games_sales;

## **Verificacion del Tipo de TABLA**

In [0]:
DESCRIBE TABLE video_games_sales